# NullVector Progress Notebook

Phase F-J / major-changes-v2 / Specs 08-09: Tree Thinning And Serving Compaction + Quickstart CLI

Prerequisites:
- Execute from the repository root with project dependencies available (for example via `uv run ...`).
- This walkthrough uses the authored Markdown ingestion path from Spec 05, deterministic noop-backed summaries from the Phase 03 gateway, retrieval corpus construction, and the serving-tree compaction stage from Spec 08.
- Spec 09 also adds an offline quickstart script at `scripts/nullvector_quickstart.py`; this notebook keeps the canonical smoke flow library-based and only points to the CLI as an optional manual path.
- The notebook writes temporary artifacts under `.artifacts/progress-spec08/` and can be re-run safely.


### Environment

This walkthrough acquires authored Markdown, builds a canonical summarized tree, builds a retrieval corpus, compacts the tree into a serving-oriented representation, and expands one serving node back to canonical retrieval evidence.

The smoke path is deterministic: the only gateway use is the noop adapter for repeatable summaries, so the compaction artifacts and evidence mapping can be inspected offline. This remediation pass also makes the canonical compaction manifest path explicit and preserves Markdown fence terminators as structural markers rather than emitted content. The quickstart CLI remains an optional manual shortcut for the offline acquisition -> tree -> retrieval flow and is intentionally not the canonical notebook execution path.


In [ ]:
# environment setup
import shutil
from pathlib import Path

ROOT = Path.cwd()
ARTIFACT_ROOT = ROOT / ".artifacts" / "progress-spec08"
if ARTIFACT_ROOT.exists():
    shutil.rmtree(ARTIFACT_ROOT)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
SOURCE_PATH = ARTIFACT_ROOT / "serving-compaction-demo.md"


In [ ]:
# imports
import json

from nullvector.domain.ledger import AcquisitionRequest, SourceDocumentKind
from nullvector.domain.retrieval import RetrievalUnitType
from nullvector.domain.tree import TreeBuildRequest, TreeCompactionRequest, TreeCompactionSettings
from nullvector.ingest.acquisition_service import AcquisitionService
from nullvector.llm import (
    GatewayAuditConfig,
    GatewayConfig,
    GatewayService,
    NoopProviderAdapter,
    NoopScriptedResponse,
)
from nullvector.retrieval import RetrievalCorpusBuilder, load_retrieval_corpus
from nullvector.tree import (
    TreeCompactionService,
    build_tree,
    expand_serving_node_ids_to_canonical_node_ids,
    load_compacted_node_mappings,
    load_compacted_tree,
)


In [ ]:
# configuration
ACQUISITION_RUN_ID = "progress-spec08-acquire"
TREE_RUN_ID = "progress-spec08-tree"
COMPACTION_RUN_ID = "progress-spec08-compaction"
MAX_CHILDREN_PER_NODE = 2

MARKDOWN_TEXT = """# Operating Handbook
Root overview for the handbook.

## Alpha Policies
Alpha policy details and revenue guidance.

## Beta Policies
Beta policy details and litigation guidance.

## Gamma Policies
Gamma policy details and deadlines.

## Delta Policies
Delta policy details and controls.

## Epsilon Policies
Epsilon policy details and escalations.
"""

SOURCE_PATH.write_text(MARKDOWN_TEXT, encoding="utf-8")
ACQUISITION_ROOT = ARTIFACT_ROOT / "acquisition"

gateway = GatewayService(
    GatewayConfig(
        default_model="test-model",
        audit=GatewayAuditConfig(persist_root=str(ARTIFACT_ROOT / "audit")),
    ),
    provider_adapter=NoopProviderAdapter(
        {
            "summarize_leaf_node": NoopScriptedResponse(
                output_json={"summary": "leaf summary", "keywords": ["leaf"]}
            ),
            "summarize_parent_node": NoopScriptedResponse(
                output_json={"summary": "parent summary", "keywords": ["parent"]}
            ),
        }
    ),
)


In [ ]:
# execution
acquisition_manifest = AcquisitionService().acquire(
    AcquisitionRequest(
        source_path=str(SOURCE_PATH),
        acquisition_run_id=ACQUISITION_RUN_ID,
        artifact_root=str(ACQUISITION_ROOT),
        source_kind=SourceDocumentKind.MARKDOWN,
        provider_identity="markdown_native",
    )
)
assert acquisition_manifest.artifact_root is not None
acquisition_manifest_path = str(Path(acquisition_manifest.artifact_root) / "manifest.json")

tree_manifest = build_tree(
    TreeBuildRequest(
        acquisition_manifest_path=acquisition_manifest_path,
        tree_run_id=TREE_RUN_ID,
        summarize=True,
    ),
    gateway=gateway,
)
assert tree_manifest.artifact_root is not None
tree_manifest_path = str(Path(tree_manifest.artifact_root) / "manifest.json")

retrieval_manifest = RetrievalCorpusBuilder().build(
    acquisition_manifest_path=acquisition_manifest_path,
    tree_manifest_path=tree_manifest_path,
)

compaction_manifest = TreeCompactionService().compact(
    TreeCompactionRequest(
        tree_manifest_path=tree_manifest_path,
        compaction_run_id=COMPACTION_RUN_ID,
        settings=TreeCompactionSettings(max_children_per_node=MAX_CHILDREN_PER_NODE),
    )
)
assert compaction_manifest.artifact_root is not None
compaction_manifest_path = str(Path(compaction_manifest.artifact_root) / "manifest.json")


In [ ]:
# execution
compacted_tree = load_compacted_tree(compaction_manifest.compacted_tree_path)
node_mappings = load_compacted_node_mappings(compaction_manifest.node_mapping_path)
retrieval_corpus = load_retrieval_corpus(retrieval_manifest.corpus_path)

root_node = next(node for node in compacted_tree if node.level == 1)
merged_node = next(
    node
    for node in compacted_tree
    if node.serving_node_id.startswith(f"{root_node.serving_node_id}::compact::")
)
canonical_node_ids = expand_serving_node_ids_to_canonical_node_ids(
    (merged_node.serving_node_id,),
    node_mappings,
)
mapped_evidence_ids = [
    unit.unit_id
    for unit in retrieval_corpus.units
    if unit.node_id in canonical_node_ids
    and unit.unit_type in (RetrievalUnitType.NODE_TEXT, RetrievalUnitType.NODE_SUMMARY)
]


In [ ]:
# inspect results
inspection = {
    "canonical_tree": {
        "tree_run_id": tree_manifest.tree_run_id,
        "committed_node_count": tree_manifest.committed_node_count,
        "node_summaries_path": tree_manifest.node_summaries_path,
    },
    "compaction": {
        "compaction_run_id": compaction_manifest.compaction_run_id,
        "manifest_path": compaction_manifest_path,
        "compacted_tree_path": compaction_manifest.compacted_tree_path,
        "node_mapping_path": compaction_manifest.node_mapping_path,
        "serving_node_count": len(compacted_tree),
        "root_child_count": len(root_node.child_serving_node_ids),
    },
    "mapping_example": {
        "serving_node_id": merged_node.serving_node_id,
        "canonical_node_ids": canonical_node_ids,
        "retrieval_evidence_ids": mapped_evidence_ids,
    },
}

print(json.dumps(inspection, indent=2))


### Known Limitations

- V1 compaction is deterministic only and does not call the gateway beyond the noop-backed tree summarization used to make the demo reproducible.
- The compacted serving tree is a derived artifact only; existing retrieval and tree-search runtimes do not consume it automatically yet.
- Merged serving nodes rely on existing canonical summaries. With `require_summaries=True`, compaction will fail instead of inventing replacement summaries for collapsed ranges.
- This remediation pass leaves PostgreSQL connection pooling deferred; the safety fixes here focus on schema validation, run-table coverage, and stable artifact/runtime behavior.
- The new quickstart CLI is intentionally offline-only in v1. Tree summarization and document-description stages still belong to the library APIs and notebook flows until a dedicated gateway bootstrap helper exists.

Optional manual quickstart example:

```bash
uv run python scripts/nullvector_quickstart.py \
  --source-path /path/to/authored.md \
  --source-kind markdown \
  --build-retrieval \
  --print-tree-summary
```
